In [2]:
from __future__ import annotations

import io
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import soundfile as sf
import resampy
from datasets import Audio, load_dataset


@dataclass(frozen=True)
class PrepConfig:
    project_root: Path = Path('/home/anna/python/MIPT/speach_recognition/FP')
    prepared_root: Path = project_root / 'data' / 'prepared_data'
    prepared_audio_dir: Path = prepared_root / 'prepared_audio'

    prepared_manifest_src: Path = prepared_root / 'prepared_manifest.tsv'
    prepared_manifest_whisper: Path = prepared_root / 'prepared_manifest_whisper.tsv'

    cc0_root: Path = project_root / 'data' / 'row_data' / 'cc0_bulgarian_speech_shunya'
    cv_root: Path = project_root / 'data' / 'row_data' / 'mozilla_common_voice_bg'
    vox_root: Path = project_root / 'data' / 'row_data' / 'voxforge_bulgarian'

    out_cc0_manifest: Path = prepared_root / 'cc0_bg_whisper.tsv'
    out_cv_manifest: Path = prepared_root / 'common_voice_bg_whisper.tsv'
    out_vox_manifest: Path = prepared_root / 'voxforge_bg_whisper.tsv'
    out_merged_manifest: Path = prepared_root / 'all_real_bg_whisper.tsv'

    target_sr: int = 16000


CFG = PrepConfig()
CFG.prepared_audio_dir.mkdir(parents=True, exist_ok=True)

In [3]:
def normalize_text(text: object) -> str:
    if text is None:
        return ''
    text = str(text)
    text = ' '.join(text.replace('\n', ' ').replace('\t', ' ').split())
    return text.strip()


def read_manifest_flexible(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep='\t')
    cols = {c.lower(): c for c in df.columns}

    if 'path' in cols:
        path_col = cols['path']
    elif 'audio' in cols:
        path_col = cols['audio']
    else:
        raise ValueError(f'{path} must contain either path or audio column')

    if 'transcription' not in cols:
        raise ValueError(f'{path} must contain transcription column')

    out = pd.DataFrame({
        'path': df[path_col].astype(str),
        'transcription': df[cols['transcription']].map(normalize_text),
    })
    out = out[out['path'].str.len() > 0]
    out = out[out['transcription'].str.len() > 0]
    return out


def write_manifest(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df[['path', 'transcription']].to_csv(path, sep='\t', index=False)


def read_audio_as_mono(path: Path) -> Tuple[np.ndarray, int]:
    audio, sr = sf.read(str(path), always_2d=False)
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)
    return audio.astype(np.float32), int(sr)


def resample_if_needed(audio: np.ndarray, sr: int, target_sr: int) -> np.ndarray:
    if sr == target_sr:
        return audio
    return resampy.resample(audio, sr, target_sr).astype(np.float32)


def ensure_16k_mono_wav(src_path: Path, dst_path: Path, target_sr: int) -> None:
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    if dst_path.exists():
        try:
            info = sf.info(str(dst_path))
            if info.samplerate == target_sr and info.channels == 1 and info.format == 'WAV':
                return
        except Exception:
            pass

    audio, sr = read_audio_as_mono(src_path)
    audio = resample_if_needed(audio, sr, target_sr)
    sf.write(str(dst_path), audio, target_sr, subtype='PCM_16')


def find_text_column(columns: Iterable[str]) -> str:
    candidates = ['transcription', 'sentence', 'text', 'transcript', 'normalized_text']
    lower_to_orig = {c.lower(): c for c in columns}
    for c in candidates:
        if c in lower_to_orig:
            return lower_to_orig[c]
    raise ValueError(f'Could not infer text column. Available columns: {list(columns)}')


def deduplicate_manifest(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Handle empty input safely (e.g. dataset split with no valid pairs).
    if df.empty and not {'path', 'transcription'}.issubset(df.columns):
        return pd.DataFrame(columns=['path', 'transcription'])

    missing = {'path', 'transcription'} - set(df.columns)
    if missing:
        raise ValueError(f'Manifest is missing required columns: {sorted(missing)}')

    df['path'] = df['path'].astype(str)
    df['transcription'] = df['transcription'].map(normalize_text)
    df = df[df['path'].str.len() > 0]
    df = df[df['transcription'].str.len() > 0]
    df = df.drop_duplicates(subset=['path'], keep='first')
    return df.reset_index(drop=True)

In [4]:
def prepare_existing_prepared_manifest(cfg: PrepConfig) -> pd.DataFrame:
    df = read_manifest_flexible(cfg.prepared_manifest_src)
    df = deduplicate_manifest(df)
    write_manifest(df, cfg.prepared_manifest_whisper)
    return df


def _decode_audio_obj(audio_obj: dict) -> Tuple[np.ndarray, int]:
    raw = audio_obj.get('bytes', None)
    if raw is not None:
        arr, sr = sf.read(io.BytesIO(raw), always_2d=False)
    else:
        p = audio_obj.get('path', None)
        if p is None:
            raise ValueError('audio object has neither bytes nor path')
        arr, sr = sf.read(str(p), always_2d=False)

    if arr.ndim == 2:
        arr = arr.mean(axis=1)
    return np.asarray(arr, dtype=np.float32), int(sr)


def prepare_parquet_dataset(
    dataset_root: Path,
    out_manifest_path: Path,
    out_audio_prefix: str,
    cfg: PrepConfig,
    ) -> pd.DataFrame:
    parquet_files = sorted(dataset_root.glob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'No parquet files found in {dataset_root}')

    all_rows = []
    for p in parquet_files:
        split = p.name.split('-')[0]
        shard_id = p.stem.replace('-', '_')

        # Load each parquet independently to avoid schema collisions across files.
        ds = load_dataset('parquet', data_files={'train': [str(p)]}, split='train')
        if 'audio' not in ds.column_names:
            raise ValueError(f'{p} has no audio column')

        text_col = find_text_column(ds.column_names)
        # Keep raw audio payload and decode with soundfile; this bypasses torchcodec.
        ds = ds.cast_column('audio', Audio(sampling_rate=None, decode=False))

        for i, ex in enumerate(ds):
            txt = normalize_text(ex.get(text_col, ''))
            if not txt:
                continue

            audio_obj = ex['audio']
            if audio_obj is None:
                continue

            try:
                arr, sr = _decode_audio_obj(audio_obj)
            except Exception:
                continue

            arr = resample_if_needed(arr, sr, cfg.target_sr)

            # Include parquet shard id to prevent train/test shard collisions.
            name = f'{out_audio_prefix}_{shard_id}_{i:07d}.wav'
            dst = cfg.prepared_audio_dir / name
            sf.write(str(dst), arr, cfg.target_sr, subtype='PCM_16')
            all_rows.append({'path': str(dst.resolve()), 'transcription': txt})

    out_df = deduplicate_manifest(pd.DataFrame(all_rows))
    write_manifest(out_df, out_manifest_path)
    return out_df


def load_voxforge_prompts(prompts_path: Path) -> Dict[str, str]:
    mapping: Dict[str, str] = {}
    if not prompts_path.exists():
        return mapping

    for line in prompts_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) < 2:
            continue
        utt_id, txt = parts
        txt = normalize_text(txt)
        if txt:
            mapping[utt_id] = txt
    return mapping


def prepare_voxforge_dataset(cfg: PrepConfig) -> pd.DataFrame:
    rows = []

    for speaker_dir in sorted(cfg.vox_root.iterdir()):
        if not speaker_dir.is_dir():
            continue

        prompts = load_voxforge_prompts(speaker_dir / 'etc' / 'PROMPTS')
        wav_dir = speaker_dir / 'wav'
        if not wav_dir.exists():
            continue

        for wav in sorted(wav_dir.glob('*.wav')):
            utt_id = wav.stem
            txt = prompts.get(utt_id, '')
            if not txt:
                continue

            dst_name = f"voxforge_{speaker_dir.name}_{utt_id}.wav"
            dst = cfg.prepared_audio_dir / dst_name
            ensure_16k_mono_wav(wav, dst, cfg.target_sr)
            rows.append({'path': str(dst.resolve()), 'transcription': txt})

    out_df = deduplicate_manifest(pd.DataFrame(rows))
    write_manifest(out_df, cfg.out_vox_manifest)
    return out_df

In [4]:
prepared_df = prepare_existing_prepared_manifest(CFG)
cc0_df = prepare_parquet_dataset(
    dataset_root=CFG.cc0_root,
    out_manifest_path=CFG.out_cc0_manifest,
    out_audio_prefix='cc0_bg',
    cfg=CFG,
)
cv_df = prepare_parquet_dataset(
    dataset_root=CFG.cv_root,
    out_manifest_path=CFG.out_cv_manifest,
    out_audio_prefix='cv_bg',
    cfg=CFG,
)
vox_df = prepare_voxforge_dataset(CFG)

merged = pd.concat([prepared_df, cc0_df, cv_df, vox_df], ignore_index=True)
merged = deduplicate_manifest(merged)
write_manifest(merged, CFG.out_merged_manifest)

print('Prepared manifests:')
print('-', CFG.prepared_manifest_whisper)
print('-', CFG.out_cc0_manifest)
print('-', CFG.out_cv_manifest)
print('-', CFG.out_vox_manifest)
print('-', CFG.out_merged_manifest)
print()
print('Rows:')
print('prepared_existing =', len(prepared_df))
print('cc0 =', len(cc0_df))
print('common_voice =', len(cv_df))
print('voxforge =', len(vox_df))
print('merged =', len(merged))

Prepared manifests:
- /home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_manifest_whisper.tsv
- /home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/cc0_bg_whisper.tsv
- /home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/common_voice_bg_whisper.tsv
- /home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/voxforge_bg_whisper.tsv
- /home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/all_real_bg_whisper.tsv

Rows:
prepared_existing = 274
cc0 = 7794
common_voice = 8206
voxforge = 0
merged = 16274


In [18]:
import re
import pandas as pd

manifest_path = "/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/all_real_bg_whisper.tsv"
pattern = re.compile(r"[0-9%]")

df = pd.read_csv(manifest_path, sep="\t")

# На случай NaN/None
df["transcription"] = df["transcription"].fillna("").astype(str)

bad = df[df["transcription"].str.contains(pattern, regex=True)]

print(f"Найдено строк: {len(bad)}")
print("-" * 80)

for i, row in bad.iterrows():
    print(f"[{i}] {row['transcription']}")

Найдено строк: 0
--------------------------------------------------------------------------------


In [5]:
# Rebuild only CC0 with collision-safe naming and refresh merged manifest.
prepared_df = read_manifest_flexible(CFG.prepared_manifest_whisper)
cv_df = read_manifest_flexible(CFG.out_cv_manifest)
vox_df = read_manifest_flexible(CFG.out_vox_manifest)

cc0_df = prepare_parquet_dataset(
    dataset_root=CFG.cc0_root,
    out_manifest_path=CFG.out_cc0_manifest,
    out_audio_prefix='cc0_bg',
    cfg=CFG,
 )

merged = pd.concat([prepared_df, cc0_df, cv_df, vox_df], ignore_index=True)
merged = deduplicate_manifest(merged)
write_manifest(merged, CFG.out_merged_manifest)

print('Rebuilt manifests:')
print('-', CFG.out_cc0_manifest)
print('-', CFG.out_merged_manifest)
print()
print('Rows:')
print('prepared_existing =', len(prepared_df))
print('cc0 =', len(cc0_df))
print('common_voice =', len(cv_df))
print('voxforge =', len(vox_df))
print('merged =', len(merged))

Rebuilt manifests:
- /home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/cc0_bg_whisper.tsv
- /home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/all_real_bg_whisper.tsv

Rows:
prepared_existing = 274
cc0 = 7794
common_voice = 8206
voxforge = 0
merged = 16274


убираю дубликаты

In [6]:
import csv
from pathlib import Path
from datetime import datetime

manifest_path = Path("/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/all_real_bg_whisper.tsv")

# backup before changes
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = manifest_path.with_name(f"{manifest_path.stem}.bak_dedup_transcription_{ts}{manifest_path.suffix}")
backup_path.write_bytes(manifest_path.read_bytes())

seen = set()
kept_rows = []
removed = 0

with manifest_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    fieldnames = reader.fieldnames
    for row in reader:
        t = (row.get("transcription") or "").strip()
        if t in seen:
            removed += 1
            continue
        seen.add(t)
        kept_rows.append(row)

with manifest_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    writer.writerows(kept_rows)

print("Done")
print("Backup:", backup_path)
print("Kept rows:", len(kept_rows))
print("Removed duplicates:", removed)

Done
Backup: /home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/all_real_bg_whisper.bak_dedup_transcription_20260716_214120.tsv
Kept rows: 9978
Removed duplicates: 6296


In [7]:
# (опционально) быстро проверить, что полных дублей транскрипции больше нет
import pandas as pd

df = pd.read_csv("/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/all_real_bg_whisper.tsv", sep="\t")
dups = df["transcription"].duplicated().sum()
print("Remaining duplicated transcriptions:", int(dups))

Remaining duplicated transcriptions: 0
